# Cell Tracking — Refined Multi-Scale DoG + Division Detection

**Key improvements over baseline:**
- Multi-scale Difference-of-Gaussians (DoG) blob detection at full resolution
- Anisotropic filtering respecting Z/XY voxel ratio (4:1)
- Intensity-weighted sub-voxel centroid refinement
- KD-tree accelerated Hungarian linking
- Gap closing (links through 1-frame detection failures)
- Division detection (parent → 2 children)
- Node count control to avoid Adjusted Jaccard penalty

In [1]:
import json
import os
import time

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import (
    gaussian_filter,
    maximum_filter,
    center_of_mass,
    label,
)
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

In [ ]:
# ============================================================
# Configuration
# ============================================================
TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'

# Physical voxel scale (µm per voxel)
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X

# Anisotropy ratio: Z is 4x coarser than XY
ANISO_RATIO = SCALE[0] / SCALE[1]  # = 4.0

# --- Detection parameters ---
# DoG sigma pairs (in voxels, accounting for anisotropy)
DOG_SIGMAS_XY = [2.0, 3.0, 4.5]  # in XY voxels
DOG_RATIO = 1.6  # sigma_large = sigma_small * DOG_RATIO

# Minimum peak separation for non-maximum suppression (in XY voxels)
NMS_SIZE_XY = 5
NMS_SIZE_Z = 2

# Centroid refinement neighbourhood radius (in voxels)
REFINE_RADIUS_XY = 4
REFINE_RADIUS_Z = 1

# --- Linking parameters ---
MAX_LINK_DISTANCE = 12.0   # um -- max distance for frame-to-frame linking
GAP_LINK_DISTANCE = 15.0   # um -- max distance for gap closing (skip 1 frame)
GAP_FRAMES = 1

# --- Division parameters ---
DIVISION_DISTANCE = 18.0   # um -- max parent-child distance for division
MIN_TRACK_LEN_DIVISION = 2

# --- Node count control ---
BASE_THRESHOLD_PERCENTILE = 85

print('Configuration loaded.')

In [ ]:
# ============================================================
# Detection: Multi-Scale DoG + Local Maxima + Centroid Refinement
# ============================================================

def multi_scale_dog(vol, sigmas_xy, dog_ratio, aniso_ratio):
    """Compute multi-scale Difference-of-Gaussians response.
    Returns the maximum DoG response across all scales.
    Uses anisotropic sigmas: sigma_z = sigma_xy / aniso_ratio."""
    vol_f = vol.astype(np.float32)
    dog_max = np.zeros_like(vol_f)
    for sigma_xy in sigmas_xy:
        sigma_z = sigma_xy / aniso_ratio
        sigma_small = (sigma_z, sigma_xy, sigma_xy)
        sigma_large = (sigma_z * dog_ratio, sigma_xy * dog_ratio, sigma_xy * dog_ratio)
        g_small = gaussian_filter(vol_f, sigma=sigma_small)
        g_large = gaussian_filter(vol_f, sigma=sigma_large)
        dog = g_small - g_large
        dog_max = np.maximum(dog_max, dog)
    return dog_max


def detect_peaks(dog_response, nms_size_z, nms_size_xy, threshold):
    """Find local maxima in the DoG response volume."""
    footprint_size = (2 * nms_size_z + 1, 2 * nms_size_xy + 1, 2 * nms_size_xy + 1)
    local_max = maximum_filter(dog_response, size=footprint_size)
    peaks_mask = (dog_response == local_max) & (dog_response > threshold)
    peak_coords = np.argwhere(peaks_mask)
    peak_values = dog_response[peaks_mask]
    return peak_coords, peak_values


def refine_centroids(vol, peak_coords, radius_z, radius_xy):
    """Refine centroid positions using intensity-weighted averaging."""
    vol_f = vol.astype(np.float32)
    Z, Y, X = vol.shape
    refined = np.empty((len(peak_coords), 3), dtype=np.float64)
    for i, (pz, py, px) in enumerate(peak_coords):
        z0 = max(0, pz - radius_z)
        z1 = min(Z, pz + radius_z + 1)
        y0 = max(0, py - radius_xy)
        y1 = min(Y, py + radius_xy + 1)
        x0 = max(0, px - radius_xy)
        x1 = min(X, px + radius_xy + 1)
        patch = vol_f[z0:z1, y0:y1, x0:x1]
        patch = np.maximum(patch - patch.min(), 0)
        total = patch.sum()
        if total > 0:
            zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
            refined[i, 0] = (zz * patch).sum() / total
            refined[i, 1] = (yy * patch).sum() / total
            refined[i, 2] = (xx * patch).sum() / total
        else:
            refined[i] = peak_coords[i].astype(np.float64)
    return refined


def detect_cells(vol, target_count=None):
    """Full detection pipeline for a single 3D volume.
    Returns refined centroids as float array (N, 3) in voxel coordinates.
    If target_count is provided, adaptively adjusts threshold."""
    dog = multi_scale_dog(vol, DOG_SIGMAS_XY, DOG_RATIO, ANISO_RATIO)
    dog_positive = dog[dog > 0]
    if len(dog_positive) == 0:
        return np.empty((0, 3), dtype=np.float64)
    
    if target_count is not None and target_count > 0:
        lo_pct, hi_pct = 50.0, 99.5
        best_coords = None
        best_diff = float('inf')
        for _ in range(10):
            mid_pct = (lo_pct + hi_pct) / 2.0
            thresh = np.percentile(dog_positive, mid_pct)
            coords, _ = detect_peaks(dog, NMS_SIZE_Z, NMS_SIZE_XY, thresh)
            n = len(coords)
            diff = abs(n - target_count)
            if diff < best_diff:
                best_diff = diff
                best_coords = coords
            if n > target_count * 1.1:
                lo_pct = mid_pct
            elif n < target_count * 0.9:
                hi_pct = mid_pct
            else:
                break
        peak_coords = best_coords
    else:
        threshold = np.percentile(dog_positive, BASE_THRESHOLD_PERCENTILE)
        peak_coords, _ = detect_peaks(dog, NMS_SIZE_Z, NMS_SIZE_XY, threshold)
    
    if len(peak_coords) == 0:
        return np.empty((0, 3), dtype=np.float64)
    
    refined = refine_centroids(vol, peak_coords, REFINE_RADIUS_Z, REFINE_RADIUS_XY)
    return refined


print('Detection functions defined.')

In [ ]:
# ============================================================
# Linking: KD-Tree Hungarian + Gap Closing + Division Detection
# ============================================================

def link_frames_hungarian(prev_phys, curr_phys, prev_ids, curr_ids, max_dist):
    """Link nodes between two frames using Hungarian assignment.
    Returns: edges, matched_prev set, matched_curr set."""
    if len(prev_phys) == 0 or len(curr_phys) == 0:
        return [], set(), set()
    n_prev = len(prev_phys)
    n_curr = len(curr_phys)
    if n_prev * n_curr < 4_000_000:
        dist = np.linalg.norm(
            prev_phys[:, None, :] - curr_phys[None, :, :], axis=2
        )
        dist[dist > max_dist] = 1e6
        row_ind, col_ind = linear_sum_assignment(dist)
        edges = []
        matched_prev = set()
        matched_curr = set()
        for ri, ci in zip(row_ind, col_ind):
            if dist[ri, ci] <= max_dist:
                edges.append((prev_ids[ri], curr_ids[ci]))
                matched_prev.add(prev_ids[ri])
                matched_curr.add(curr_ids[ci])
        return edges, matched_prev, matched_curr
    else:
        tree = cKDTree(curr_phys)
        neighbors = tree.query_ball_point(prev_phys, r=max_dist)
        row_list, col_list, dist_list = [], [], []
        for i, nbrs in enumerate(neighbors):
            for j in nbrs:
                d = np.linalg.norm(prev_phys[i] - curr_phys[j])
                row_list.append(i)
                col_list.append(j)
                dist_list.append(d)
        if not row_list:
            return [], set(), set()
        unique_rows = sorted(set(row_list))
        unique_cols = sorted(set(col_list))
        row_map = {r: i for i, r in enumerate(unique_rows)}
        col_map = {c: i for i, c in enumerate(unique_cols)}
        cost = np.full((len(unique_rows), len(unique_cols)), 1e6)
        for r, c, d in zip(row_list, col_list, dist_list):
            cost[row_map[r], col_map[c]] = d
        ri, ci = linear_sum_assignment(cost)
        edges = []
        matched_prev = set()
        matched_curr = set()
        for r, c in zip(ri, ci):
            if cost[r, c] <= max_dist:
                orig_r = unique_rows[r]
                orig_c = unique_cols[c]
                edges.append((prev_ids[orig_r], curr_ids[orig_c]))
                matched_prev.add(prev_ids[orig_r])
                matched_curr.add(curr_ids[orig_c])
        return edges, matched_prev, matched_curr


def detect_divisions(parent_phys, parent_ids, child_phys, child_ids,
                     matched_parent_ids, matched_child_ids,
                     max_dist, track_lengths):
    """Detect cell divisions: unmatched parent -> exactly 2 unmatched children."""
    division_edges = []
    unmatched_parent_mask = np.array(
        [pid not in matched_parent_ids for pid in parent_ids]
    )
    unmatched_child_mask = np.array(
        [cid not in matched_child_ids for cid in child_ids]
    )
    if not np.any(unmatched_parent_mask) or np.sum(unmatched_child_mask) < 2:
        return division_edges
    unmatched_parent_phys = parent_phys[unmatched_parent_mask]
    unmatched_parent_ids = [pid for pid, m in zip(parent_ids, unmatched_parent_mask) if m]
    unmatched_child_phys = child_phys[unmatched_child_mask]
    unmatched_child_ids = [cid for cid, m in zip(child_ids, unmatched_child_mask) if m]
    if len(unmatched_child_phys) < 2:
        return division_edges
    child_tree = cKDTree(unmatched_child_phys)
    used_children = set()
    for i, pid in enumerate(unmatched_parent_ids):
        if track_lengths.get(pid, 0) < MIN_TRACK_LEN_DIVISION:
            continue
        nearby = child_tree.query_ball_point(unmatched_parent_phys[i], r=max_dist)
        # Filter out already-used children
        nearby = [n for n in nearby if unmatched_child_ids[n] not in used_children]
        if len(nearby) == 2:
            c1_phys = unmatched_child_phys[nearby[0]]
            c2_phys = unmatched_child_phys[nearby[1]]
            sister_dist = np.linalg.norm(c1_phys - c2_phys)
            if sister_dist < max_dist * 1.5:
                cid1 = unmatched_child_ids[nearby[0]]
                cid2 = unmatched_child_ids[nearby[1]]
                division_edges.append((pid, cid1))
                division_edges.append((pid, cid2))
                used_children.add(cid1)
                used_children.add(cid2)
    return division_edges


def gap_close(lost_tracks, curr_phys, curr_ids, matched_curr_ids,
              max_dist, scale):
    """Attempt to reconnect tracks lost for 1 frame."""
    gap_edges = []
    reconnected_ids = set()
    unmatched_mask = np.array(
        [cid not in matched_curr_ids for cid in curr_ids]
    )
    if not np.any(unmatched_mask) or not lost_tracks:
        return gap_edges, reconnected_ids
    unmatched_curr_phys = curr_phys[unmatched_mask]
    unmatched_curr_ids = [cid for cid, m in zip(curr_ids, unmatched_mask) if m]
    if len(unmatched_curr_phys) == 0:
        return gap_edges, reconnected_ids
    lost_ids = list(lost_tracks.keys())
    lost_phys = np.array([lost_tracks[lid][0] for lid in lost_ids])
    if len(lost_phys) == 0:
        return gap_edges, reconnected_ids
    edges, matched_lost, _ = link_frames_hungarian(
        lost_phys, unmatched_curr_phys,
        lost_ids, unmatched_curr_ids,
        max_dist
    )
    return edges, matched_lost


print('Linking functions defined.')

In [ ]:
# ============================================================
# Main Processing Loop
# ============================================================

test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr')
)

all_rows = []
total_start = time.time()

for folder_name in test_folder_names:
    sample_start = time.time()
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')

    # --- Read array metadata ---
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        arr_meta = json.load(f)
    shape = tuple(arr_meta['shape'])  # (T, Z, Y, X)
    dtype = np.dtype(arr_meta['data_type'])
    n_t = shape[0]
    vol_shape = shape[1:]  # (Z, Y, X)

    # --- Calibration: detect on first frame to estimate cell density ---
    chunk_path = os.path.join(zarr_path, '0', 'c', '0', '0', '0', '0')
    with open(chunk_path, 'rb') as fh:
        compressed = fh.read()
    decompressed = blosc2.decompress(compressed)
    vol0 = np.frombuffer(decompressed, dtype=dtype).reshape(vol_shape)
    calib_centroids = detect_cells(vol0, target_count=None)
    cells_per_frame_estimate = max(len(calib_centroids), 10)

    # --- Process all timepoints ---
    node_id_counter = 1
    frame_phys = {}     # t -> dict {node_id: physical_coords}
    track_lengths = {}  # node_id -> track length
    lost_tracks = {}    # node_id -> (physical_coords, frames_lost)
    sample_edges = []
    sample_nodes = []

    for t in range(n_t):
        # --- Load volume ---
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as fh:
            compressed = fh.read()
        decompressed = blosc2.decompress(compressed)
        vol = np.frombuffer(decompressed, dtype=dtype).reshape(vol_shape)

        # --- Detect cells ---
        centroids_voxel = detect_cells(vol, target_count=cells_per_frame_estimate)

        # --- Create nodes ---
        curr_ids = []
        curr_phys_dict = {}
        for cent in centroids_voxel:
            nid = node_id_counter
            node_id_counter += 1
            z_int = max(0, min(vol_shape[0] - 1, int(round(cent[0]))))
            y_int = max(0, min(vol_shape[1] - 1, int(round(cent[1]))))
            x_int = max(0, min(vol_shape[2] - 1, int(round(cent[2]))))
            curr_ids.append(nid)
            curr_phys_dict[nid] = cent * SCALE
            track_lengths[nid] = 1
            sample_nodes.append({
                'dataset': folder_name,
                'row_type': 'node',
                'node_id': nid,
                't': t,
                'z': z_int,
                'y': y_int,
                'x': x_int,
                'source_id': -1,
                'target_id': -1,
            })

        frame_phys[t] = curr_phys_dict

        # --- Linking ---
        if t > 0 and (t - 1) in frame_phys:
            prev_phys_dict = frame_phys[t - 1]
            prev_ids = list(prev_phys_dict.keys())
            if prev_ids and curr_ids:
                prev_phys_arr = np.array([prev_phys_dict[pid] for pid in prev_ids])
                curr_phys_arr = np.array([curr_phys_dict[cid] for cid in curr_ids])

                # Frame-to-frame linking
                edges, matched_prev, matched_curr = link_frames_hungarian(
                    prev_phys_arr, curr_phys_arr,
                    prev_ids, curr_ids,
                    MAX_LINK_DISTANCE
                )
                for src, tgt in edges:
                    track_lengths[tgt] = track_lengths.get(src, 1) + 1
                    sample_edges.append({
                        'dataset': folder_name,
                        'row_type': 'edge',
                        'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                        'source_id': src, 'target_id': tgt,
                    })

                # --- Gap Closing ---
                if lost_tracks:
                    gap_edges, reconnected = gap_close(
                        lost_tracks, curr_phys_arr, curr_ids,
                        matched_curr, GAP_LINK_DISTANCE, SCALE
                    )
                    for src, tgt in gap_edges:
                        track_lengths[tgt] = track_lengths.get(src, 1) + 1
                        sample_edges.append({
                            'dataset': folder_name,
                            'row_type': 'edge',
                            'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                            'source_id': src, 'target_id': tgt,
                        })
                        matched_curr.add(tgt)
                    for rid in reconnected:
                        lost_tracks.pop(rid, None)

                # --- Division Detection ---
                div_edges = detect_divisions(
                    prev_phys_arr, prev_ids,
                    curr_phys_arr, curr_ids,
                    matched_prev, matched_curr,
                    DIVISION_DISTANCE, track_lengths
                )
                for src, tgt in div_edges:
                    track_lengths[tgt] = 1
                    sample_edges.append({
                        'dataset': folder_name,
                        'row_type': 'edge',
                        'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                        'source_id': src, 'target_id': tgt,
                    })

                # --- Update lost tracks ---
                new_lost = {}
                for pid in prev_ids:
                    if pid not in matched_prev:
                        in_div = any(s == pid for s, _ in div_edges)
                        if not in_div:
                            new_lost[pid] = (prev_phys_dict[pid], 1)
                updated_lost = {}
                for lid, (coords, age) in lost_tracks.items():
                    if lid not in reconnected and age < GAP_FRAMES:
                        updated_lost[lid] = (coords, age + 1)
                updated_lost.update(new_lost)
                lost_tracks = updated_lost
            else:
                lost_tracks = {}

        # Update running estimate
        n_detected = len(centroids_voxel)
        if n_detected > 0:
            cells_per_frame_estimate = int(0.7 * cells_per_frame_estimate + 0.3 * n_detected)

        # Free previous frame data (keep only t-1)
        if t >= 2 and (t - 2) in frame_phys:
            del frame_phys[t - 2]

    all_rows.extend(sample_nodes)
    all_rows.extend(sample_edges)

    elapsed = time.time() - sample_start
    n_nodes = len(sample_nodes)
    n_edges = len(sample_edges)
    print(f'{folder_name}: {n_nodes} nodes, {n_edges} edges ({elapsed:.1f}s)')

total_elapsed = time.time() - total_start
print(f'\nTotal processing time: {total_elapsed:.1f}s')

In [ ]:
# ============================================================
# Create Submission
# ============================================================

submission = pd.DataFrame(all_rows)
submission = submission[['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']]
submission.index = range(len(submission))
submission.index.name = 'id'

int_cols = ['node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
for col in int_cols:
    submission[col] = submission[col].astype(int)

submission.to_csv('submission.csv')

# --- Validation report ---
n_nodes = (submission['row_type'] == 'node').sum()
n_edges = (submission['row_type'] == 'edge').sum()
n_datasets = submission['dataset'].nunique()

print(f'Submission written: {len(submission)} rows')
print(f'  Datasets: {n_datasets}')
print(f'  Nodes: {n_nodes}')
print(f'  Edges: {n_edges}')
print(f'  Edges/Node ratio: {n_edges / max(n_nodes, 1):.2f}')
print(f'\nPer-dataset summary:')
for ds in sorted(submission['dataset'].unique()):
    ds_data = submission[submission['dataset'] == ds]
    ds_nodes = (ds_data['row_type'] == 'node').sum()
    ds_edges = (ds_data['row_type'] == 'edge').sum()
    ds_t_range = ds_data[ds_data['row_type'] == 'node']['t']
    n_frames = ds_t_range.nunique() if len(ds_t_range) > 0 else 0
    avg_per_frame = ds_nodes / max(n_frames, 1)
    print(f'  {ds}: {ds_nodes} nodes ({avg_per_frame:.0f}/frame), {ds_edges} edges, {n_frames} frames')